# Outline:

Constraint insertion: 
- Min/max order quantity
- Min/max concentration (:=quality attribute)
- Min/max price

Error specifications and ranking:
- Deviation metrics (ie. KL)
- Filter out uncertain options (Toggle)


## Get attibutes from text2product

In [1]:
import importlib
import re
import sys
sys.path.insert(0, ".")

import db, text2product
importlib.reload(db)
importlib.reload(text2product)

from pathlib import Path
from text2product import html_to_text, extract_product_info
from db import upsert_supplier_product_prices, upsert_supplier_product_info, get_processed_supplier_products

SUPPLIER_PRODUCTS_DIR = Path("../data/supplier_products")
FNAME_RE = re.compile(r"^(\d+)_.+?_(\d+)_(RM-[^.]+)\.html$")

already_done = get_processed_supplier_products()
files = sorted(SUPPLIER_PRODUCTS_DIR.glob("*.html"))
print(f"Processing {len(files)} files ({len(already_done)} already done)...")

for path in files:
    m = FNAME_RE.match(path.name)
    if not m:
        print(f"  SKIP (unrecognised filename): {path.name}")
        continue

    supplier_id, product_id, sku_slug = int(m.group(1)), int(m.group(2)), m.group(3)

    if (supplier_id, product_id) in already_done:
        continue

    try:
        page_text = html_to_text(path.read_text(errors="replace"))
        result = extract_product_info(page_text, sku_slug)
    except Exception as e:
        print(f"  ERROR {path.name}: {e}")
        continue

    prices          = result.get("prices") or []
    purity          = result.get("purity")
    quality         = result.get("quality")
    quality_score   = result.get("quality_score")
    quality_metrics = result["quality_metrics"]

    if prices:
        upsert_supplier_product_prices(supplier_id, product_id, prices)
    upsert_supplier_product_info(supplier_id, product_id, purity, quality, quality_score, quality_metrics)
    already_done.add((supplier_id, product_id))

    metrics_present = [k for k, v in quality_metrics.items() if v is not None]
    print(f"  {path.name}: {len(prices)} tier(s), purity={purity}, quality={quality!r}, metrics={metrics_present}")

print("Done.")


Processing 203 files...
  12_Custom_Probiotics_283_RM-C13-probiotic-cultures-4fa0be80.html: 0 tier(s), purity=None, quality=None, metrics=[]
  24_Nutra_Blend_282_RM-C13-energy-support-botanicals-nutrients-4823efcb.html: 0 tier(s), purity=None, quality='feed grade', metrics=[]
  24_Nutra_Blend_582_RM-C33-ferment-media-c911f8c0.html: 1 tier(s), purity=None, quality='feed grade', metrics=[]
  24_Nutra_Blend_751_RM-C45-prebiotic-fiber-19375eea.html: 0 tier(s), purity='2%', quality='feed grade', metrics=[]
  24_Nutra_Blend_820_RM-C52-performance-support-nutrients-165afec4.html: 0 tier(s), purity='84% active', quality='feed grade', metrics=[]
  24_Nutra_Blend_885_RM-C57-cultured-nutrients-5c87fee4.html: 0 tier(s), purity=None, quality='feed grade', metrics=[]


KeyboardInterrupt: 

## Constraint filtering

In [ ]:
import sys
sys.path.insert(0, ".")

from db import get_supplier_products_enriched
from filter_products import filter_products, make_filters

price_range    = (None, None)
quantity_range = (None, None)
purity_range   = (None, None)
quality_range  = (None, None)  # 0–1: 0.7=food/kosher, 0.9=GMP, 1.0=USP/pharma

# Per-metric quality constraints — uncomment to activate
quality_metrics = {
    # "identity_confidence": (0.95, None),   # ≥95% match to reference
    # "assay_potency":       (0.97, 1.03),   # label claim ±3%
    # "heavy_metals":        (None, 10.0),   # ≤10 ppm aggregate
    # "pesticide_residues":  (None, 100.0),  # ≤100 ppb
    # "microbial_limits":    (1.0,  None),   # must pass (1.0)
    # "moisture_content":    (None, 0.05),   # ≤5% water content
    # "residual_solvents":   (None, 410.0),  # ≤410 ppm (ICH Class 3 limit)
}

products = get_supplier_products_enriched()
filters  = make_filters(price_range, quantity_range, purity_range, quality_range, quality_metrics)
results  = filter_products(products, filters)

print(f"{len(results)}/{len(products)} products passed")
results


Example

In [ ]:
# Pharmaceutical/USP-grade products ≥98% purity, under $50, at least 100g available
# with strict identity confirmation and heavy metals within limits
products = get_supplier_products_enriched()

filters = make_filters(
    price_range=(None, 50.0),      # max $50
    quantity_range=(100.0, None),  # at least 100g
    purity_range=(0.98, None),     # ≥98% purity
    quality_range=(0.9, None),     # GMP or better (0.9=GMP, 1.0=USP/pharma)
    quality_metrics={
        "identity_confidence": (0.95, None),  # ≥95% chromatographic/spectral match
        "assay_potency":       (0.97, 1.03),  # label claim ±3%
        "heavy_metals":        (None, 10.0),  # ≤10 ppm
        "microbial_limits":    (1.0,  None),  # must pass
    },
)

results = filter_products(products, filters)
print(f"{len(results)}/{len(products)} products passed")
results


## Ranking function

## Evaluation

- make 10 queries with manual tagged correctness
- determne correctness comparing results of model top-5 suggestions